In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D

# Enable inline plotting
%matplotlib inline

In [2]:
class OneQubitState:
    """
    A class to represent a 1-qubit quantum state on the unit circle
    |H⟩ (horizontal polarization) is at +x (1, 0)
    |V⟩ (vertical polarization) is at +y (0, 1)
    """
    def __init__(self, angle=0, phase=0):
        """
        Initialize a 1-qubit state using angle and phase
        angle: position on unit circle (0 to 2π)
        phase: phase offset for time evolution
        """
        self.angle = angle
        self.phase = phase
        self.update_state()
    
    def update_state(self):
        """Update the quantum state vector from the angle"""
        # Quantum state amplitudes (proper qubit representation)
        # At angle=0: |H⟩ = [1, 0] (horizontal)
        # At angle=π: |V⟩ = [0, 1] (vertical)
        self.amplitude_h = np.cos(self.angle/2) * np.exp(1j * self.phase)
        self.amplitude_v = np.sin(self.angle/2) * np.exp(1j * self.phase)
        
        # State vector [|H⟩, |V⟩]
        self.state = np.array([self.amplitude_h, self.amplitude_v])
        
        # Probabilities
        self.prob_h = np.abs(self.amplitude_h)**2
        self.prob_v = np.abs(self.amplitude_v)**2
        
        # Unit circle position - sync with photon polarization
        # Map qubit angle to polarization direction properly
        # For horizontal |H⟩: angle=0 → polarization along +x
        # For vertical |V⟩: angle=π → polarization along +y
        self.polarization_x = np.cos(self.angle/2)**2 - np.sin(self.angle/2)**2  # cos(angle)
        self.polarization_y = 2 * np.cos(self.angle/2) * np.sin(self.angle/2)    # sin(angle)
        
        # Unit circle position (matches polarization direction)
        self.x = self.polarization_x  # Ex component
        self.y = self.polarization_y  # Ey component
    
    def set_angle(self, angle):
        """Update the angle and recalculate the state"""
        self.angle = angle
        self.update_state()
    
    def set_phase(self, phase):
        """Update the phase and recalculate the state"""
        self.phase = phase
        self.update_state()
    
    def get_bloch_vector(self):
        """Get the position on the unit circle for visualization"""
        return np.array([self.x, self.y])

In [3]:
def create_interactive_1qubit_visualization():
    """
    Create an interactive visualization of a 1-qubit system with photon polarization
    """
    # Initialize the quantum state
    qubit_state = OneQubitState()
    
    # Create the figure with subplots: unit circle + 3D photon wave
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("1-Qubit State", "Photon Polarization"),
        specs=[[{"type": "scatter"}, {"type": "scatter3d"}]],
        horizontal_spacing=0.15
    )
    
    # Create sliders for controlling the quantum state
    angle_slider = widgets.FloatSlider(
        value=0, min=0, max=2*np.pi, step=0.05,
        description='θ:', style={'description_width': '30px'},
        layout=widgets.Layout(width='400px'),
        readout=False  # Disable default readout, we'll add custom one
    )
    
    # Custom readout for angle in terms of π
    angle_readout = widgets.HTML(value="θ = 0")
    
    time_slider = widgets.FloatSlider(
        value=0, min=0, max=4*np.pi, step=0.1,
        description='Time:', style={'description_width': '50px'},
        layout=widgets.Layout(width='400px')
    )
    
    # Output widget for the plot
    output = widgets.Output()
    
    def format_angle_as_pi_fraction(angle):
        """Convert angle to π fraction string"""
        if angle == 0:
            return "θ = 0"
        elif abs(angle - np.pi/4) < 0.05:
            return "θ = π/4"
        elif abs(angle - np.pi/2) < 0.05:
            return "θ = π/2"
        elif abs(angle - 3*np.pi/4) < 0.05:
            return "θ = 3π/4"
        elif abs(angle - np.pi) < 0.05:
            return "θ = π"
        elif abs(angle - 5*np.pi/4) < 0.05:
            return "θ = 5π/4"
        elif abs(angle - 3*np.pi/2) < 0.05:
            return "θ = 3π/2"
        elif abs(angle - 7*np.pi/4) < 0.05:
            return "θ = 7π/4"
        elif abs(angle - 2*np.pi) < 0.05:
            return "θ = 2π"
        else:
            # For other values, show as decimal fraction of π
            pi_fraction = angle / np.pi
            return f"θ = {pi_fraction:.2f}π"
    
    def update_angle_readout(*args):
        """Update the angle readout display"""
        angle_readout.value = format_angle_as_pi_fraction(angle_slider.value)
    
    def update_plot(*args):
        """Update the visualization when sliders change"""
        # Update angle readout
        update_angle_readout()
        
        with output:
            clear_output(wait=True)
            
            # Update quantum state
            qubit_state.set_angle(angle_slider.value)
            qubit_state.set_phase(time_slider.value)
            
            # Get the state vector
            state_vector = qubit_state.get_bloch_vector()
            x_state, y_state = state_vector[0], state_vector[1]
            
            # Clear previous traces
            fig.data = []
            
            # === UNIT CIRCLE PLOT ===
            # Unit circle
            circle_theta = np.linspace(0, 2*np.pi, 100)
            circle_x = np.cos(circle_theta)
            circle_y = np.sin(circle_theta)
            
            fig.add_trace(go.Scatter(
                x=circle_x, y=circle_y,
                mode='lines',
                line=dict(color='lightgray', width=2),
                name='Unit Circle',
                showlegend=False
            ), row=1, col=1)
            
            # Basis states |H⟩ and |V⟩
            fig.add_trace(go.Scatter(
                x=[1], y=[0],
                mode='markers+text',
                marker=dict(size=15, color='red', symbol='circle'),
                text=['|H⟩'],
                textposition='bottom right',
                name='Horizontal',
                showlegend=False
            ), row=1, col=1)
            
            fig.add_trace(go.Scatter(
                x=[0], y=[1],
                mode='markers+text',
                marker=dict(size=15, color='blue', symbol='circle'),
                text=['|V⟩'],
                textposition='top right',
                name='Vertical',
                showlegend=False
            ), row=1, col=1)
            
            # Current state vector
            fig.add_trace(go.Scatter(
                x=[0, x_state], y=[0, y_state],
                mode='lines+markers',
                line=dict(color='green', width=4),
                marker=dict(size=[0, 16], color=['green', 'green']),
                name='State Vector',
                showlegend=False
            ), row=1, col=1)
            
            # State point with probability info
            fig.add_trace(go.Scatter(
                x=[x_state], y=[y_state],
                mode='text',
                text=[f'|ψ⟩<br>P(H)={qubit_state.prob_h:.3f}<br>P(V)={qubit_state.prob_v:.3f}'],
                textposition='top center',
                showlegend=False
            ), row=1, col=1)
            
            # === 3D PHOTON WAVE ===
            # Create 3D sine wave representing photon
            z_wave = np.linspace(0, 4*np.pi, 200)  # Propagation direction
            t = time_slider.value
            
            # Electric field components based on polarization state
            Ex = qubit_state.polarization_x * np.sin(z_wave - t)  # X polarization
            Ey = qubit_state.polarization_y * np.sin(z_wave - t)  # Y polarization
            
            # 3D wave traces
            fig.add_trace(go.Scatter3d(
                x=Ex, y=Ey, z=z_wave,
                mode='lines',
                line=dict(color='purple', width=6),
                name='Photon Wave',
                showlegend=False
            ), row=1, col=2)
            
            # Add a moving dot on the wave (photon particle)
            photon_pos = 2*np.pi + t % (2*np.pi)  # Moving photon position
            photon_idx = np.argmin(np.abs(z_wave - photon_pos))
            
            fig.add_trace(go.Scatter3d(
                x=[Ex[photon_idx]], y=[Ey[photon_idx]], z=[z_wave[photon_idx]],
                mode='markers',
                marker=dict(size=8, color='yellow', symbol='circle'),
                name='Photon',
                showlegend=False
            ), row=1, col=2)
            
            # Add polarization lines from origin to wave (every 20th point)
            step = 20
            for i in range(0, len(z_wave), step):
                fig.add_trace(go.Scatter3d(
                    x=[0, Ex[i]], y=[0, Ey[i]], z=[z_wave[i], z_wave[i]],
                    mode='lines',
                    line=dict(color='black', width=1, dash='dot'),
                    opacity=0.8,
                    showlegend=False
                ), row=1, col=2)
            
            # Add propagation axis line (dashed black along z-axis)
            fig.add_trace(go.Scatter3d(
                x=[0, 0], y=[0, 0], z=[0, 4*np.pi],
                mode='lines',
                line=dict(color='black', width=2, dash='dash'),
                opacity=0.7,
                name='Propagation axis',
                showlegend=False
            ), row=1, col=2)
            
            # Add coordinate axes for reference
            fig.add_trace(go.Scatter3d(
                x=[-1.5, 1.5], y=[0, 0], z=[0, 0],
                mode='lines',
                line=dict(color='red', width=3),
                name='Ex axis',
                showlegend=False
            ), row=1, col=2)
            
            fig.add_trace(go.Scatter3d(
                x=[0, 0], y=[-1.5, 1.5], z=[0, 0],
                mode='lines',
                line=dict(color='blue', width=3),
                name='Ey axis',
                showlegend=False
            ), row=1, col=2)
            
            # Update layout - remove all axis labels and ticks
            fig.update_xaxes(
                title_text="", 
                range=[-1.2, 1.2], 
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                row=1, col=1
            )
            fig.update_yaxes(
                title_text="", 
                range=[-1.2, 1.2], 
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                row=1, col=1
            )
            
            fig.update_layout(
                height=600,
                title_text="",
                title_x=0.5,
                scene=dict(
                    xaxis=dict(title="", showticklabels=False, showgrid=False, zeroline=False),
                    yaxis=dict(title="", showticklabels=False, showgrid=False, zeroline=False),
                    zaxis=dict(title="", showticklabels=False, showgrid=False, zeroline=False),
                    camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
                )
            )
            
            fig.show()
    
    # Connect sliders to update function
    angle_slider.observe(update_plot, names='value')
    angle_slider.observe(update_angle_readout, names='value')
    time_slider.observe(update_plot, names='value')
    
    # Display the interface
    controls = widgets.VBox([
        widgets.HTML("<h3>Polarization</h3>"),
        widgets.HTML("<b>|H⟩ = Horizontal, |V⟩ = Vertical</b>"),
        widgets.HBox([angle_slider, angle_readout]),
        time_slider,
    ])
    
    display(controls, output)
    
    # Initial plot and angle readout
    update_angle_readout()
    update_plot()
    
    return qubit_state, fig

In [4]:
# Create and display the interactive 1-qubit visualization
print("Creating interactive 1-qubit polarization visualization...")
qubit_system, plot_fig = create_interactive_1qubit_visualization()

Creating interactive 1-qubit polarization visualization...


Output()